In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../../data/bbc-news-data.csv", sep='\t')
df.head()

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [3]:
df.shape

(2225, 4)

In [4]:
df['text'] = df['title']+ " " + df['content']

In [5]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

In [6]:
import re
import spacy
import nltk
import contractions

In [7]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'W+', ' ', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [9]:
df['clean_text'] = df['text'].apply(preprocess)
df['clean_text'].head()

0    ad sale boost time warner profit quarterly pro...
1    dollar gain greenspan speech dollar hit highes...
2    yukos unit buyer face loan claim owner embattl...
3    high fuel price hit ba 's profit british airwa...
4    pernod takeover talk lift domecq share uk drin...
Name: clean_text, dtype: str

In [10]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_tfidf = vectorizer.fit_transform(df['clean_text'])
X_tfidf.shape

(2225, 5000)

In [13]:
documents = df['clean_text'].astype(str).apply(lambda x: x.split()).tolist()

In [14]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=documents,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

In [15]:
vector = model.wv["product"]
print(vector.shape)  
print(vector[:10]) 

(100,)
[-0.1429603   0.05602834  0.19810146  0.12626043 -0.09351442 -0.02001408
 -0.17774542  0.40732062  0.13667364  0.05275366]
